<a href="https://colab.research.google.com/github/manastasijas-pixel/Netology-project/blob/main/DLL_6_%D1%80%D0%B5%D0%BA%D1%83%D1%80%D0%B5%D0%BD%D1%82%D0%BD%D1%8B%D0%B5_%D1%81%D0%B5%D1%82%D0%B8_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

In [ ]:
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
import torch
from torch import nn

import warnings
warnings.filterwarnings('ignore')

Обучите нейронную сеть решать шифр Цезаря.

Что необходимо сделать:

Написать алгоритм шифра Цезаря для генерации выборки (сдвиг на К каждой буквы. Например, при сдвиге на 2 буква “А” переходит в букву “В” и тп)
Сделать нейронную сеть
Обучить ее (вход - зашифрованная фраза, выход - дешифрованная фраза)
Проверить качество

In [ ]:
import pandas as pd  # для работы с данными
import time  # для оценки времени
import torch  # для написания нейросети

In [ ]:
#чтобы загрузить файлы с данными в блокнот/ Вызвать команду столько раз, сколько файлов надо загрузить
dates = files.upload()

Saving simpsons_script_lines.csv to simpsons_script_lines.csv


In [ ]:
df = pd.read_csv('simpsons_script_lines.csv')
df.head()

,id,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,9549,32,209,"Miss Hoover: No, actually, it was a little of ...",848000,True,464.0,3.0,Miss Hoover,Springfield Elementary School,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...,31
1,9550,32,210,Lisa Simpson: (NEAR TEARS) Where's Mr. Bergstrom?,856000,True,9.0,3.0,Lisa Simpson,Springfield Elementary School,Where's Mr. Bergstrom?,wheres mr bergstrom,3
2,9551,32,211,Miss Hoover: I don't know. Although I'd sure l...,856000,True,464.0,3.0,Miss Hoover,Springfield Elementary School,I don't know. Although I'd sure like to talk t...,i dont know although id sure like to talk to h...,22
3,9552,32,212,Lisa Simpson: That life is worth living.,864000,True,9.0,3.0,Lisa Simpson,Springfield Elementary School,That life is worth living.,that life is worth living,5
4,9553,32,213,Edna Krabappel-Flanders: The polls will be ope...,864000,True,40.0,3.0,Edna Krabappel-Flanders,Springfield Elementary School,The polls will be open from now until the end ...,the polls will be open from now until the end ...,33


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [ ]:
# вытащим все фразы и сделаем из них список
phrases = df['normalized_text'].tolist()
phrases.head()

['no actually it was a little of both sometimes when a disease is in all the magazines and all the news shows its only natural that you think you have it',
 'wheres mr bergstrom',
 'i dont know although id sure like to talk to him he didnt touch my lesson plan what did he teach you',
 'that life is worth living',
 'the polls will be open from now until the end of recess now just in case any of you have decided to put any thought into this well have our final statements martin',
 'i dont think theres anything left to say',
 'bart',
 'victory party under the slide',
 nan,
 'mr bergstrom mr bergstrom',
 'hey hey he moved out this morning he must have a new job -- he took his copernicus costume',
 'do you know where i could find him',
 'i think hes taking the next train to capital city',
 'the train how like him traditional yet environmentally sound',
 'yes and its been the backbone of our country since leland stanford drove that golden spike at promontory point',
 'i see he touched you to

In [ ]:
# алфавит, чтобы знать куда смещать
alfabet = ['none'] + list(' abcdefghijklmnopqrstuvwxyz')
# сделаем справочник - буква и ее индекс
AB_INDEX= {w: i for i, w in enumerate(alfabet)}

In [ ]:
text = [[c for c in ph] for ph in phrases if (type(ph) is str) and (set(ph).issubset(AB_INDEX))]
print(text[0])

['n', 'o', ' ', 'a', 'c', 't', 'u', 'a', 'l', 'l', 'y', ' ', 'i', 't', ' ', 'w', 'a', 's', ' ', 'a', ' ', 'l', 'i', 't', 't', 'l', 'e', ' ', 'o', 'f', ' ', 'b', 'o', 't', 'h', ' ', 's', 'o', 'm', 'e', 't', 'i', 'm', 'e', 's', ' ', 'w', 'h', 'e', 'n', ' ', 'a', ' ', 'd', 'i', 's', 'e', 'a', 's', 'e', ' ', 'i', 's', ' ', 'i', 'n', ' ', 'a', 'l', 'l', ' ', 't', 'h', 'e', ' ', 'm', 'a', 'g', 'a', 'z', 'i', 'n', 'e', 's', ' ', 'a', 'n', 'd', ' ', 'a', 'l', 'l', ' ', 't', 'h', 'e', ' ', 'n', 'e', 'w', 's', ' ', 's', 'h', 'o', 'w', 's', ' ', 'i', 't', 's', ' ', 'o', 'n', 'l', 'y', ' ', 'n', 'a', 't', 'u', 'r', 'a', 'l', ' ', 't', 'h', 'a', 't', ' ', 'y', 'o', 'u', ' ', 't', 'h', 'i', 'n', 'k', ' ', 'y', 'o', 'u', ' ', 'h', 'a', 'v', 'e', ' ', 'i', 't']


In [ ]:
# Подготовим обучающую выборку. по готовым текстам сделаем зашифрованные
MAX_LEN = 50
# два пустых массива. y  - исходный текст. X - с сдвигом
X = torch.zeros((len(text), MAX_LEN), dtype=int)
y = torch.zeros((len(text), MAX_LEN), dtype=int)
K = 11
for i in tqdm(range(len(text))):
    #K = np.random.choice(range(1, 27))
    for j, w in enumerate(text[i]):
        if j >= MAX_LEN:
            break
        idx = AB_INDEX[w]
        y[i, j] = idx
        X[i, j] = int(np.ceil(np.mod(idx + (K - 0.1), 27)))
    #y[i, 50] = K

100%|██████████| 117015/117015 [01:41<00:00, 1151.25it/s]


In [ ]:
# посмотрим на произвольное значение выборки
T = 16
print(f'Шифр Цезаря: сдвиг на {K} символов\n')
print('Исходный текст:')
print(text[T])
print()
print('Исходный текст индексами:')
print(y[T,:50])
print()
print('Зашифрованный текст индексами:')
print(X[T,:50])

Шифр Цезаря: сдвиг на 11 символов

Исходный текст:
['w', 'e', 'l', 'l', ' ', 'y', 'o', 'u', ' ', 'g', 'o', 't', ' ', 't', 'h', 'a', 't', ' ', 'r', 'i', 'g', 'h', 't', ' ', 't', 'h', 'a', 'n', 'k', 's', ' ', 'f', 'o', 'r', ' ', 'y', 'o', 'u', 'r', ' ', 'v', 'o', 't', 'e', ' ', 'g', 'i', 'r', 'l', 's']

Исходный текст индексами:
tensor([24,  6, 13, 13,  1, 26, 16, 22,  1,  8, 16, 21,  1, 21,  9,  2, 21,  1,
        19, 10,  8,  9, 21,  1, 21,  9,  2, 15, 12, 20,  1,  7, 16, 19,  1, 26,
        16, 22, 19,  1, 23, 16, 21,  6,  1,  8, 10, 19, 13, 20])

Зашифрованный текст индексами:
tensor([ 8, 17, 24, 24, 12, 10, 27,  6, 12, 19, 27,  5, 12,  5, 20, 13,  5, 12,
         3, 21, 19, 20,  5, 12,  5, 20, 13, 26, 23,  4, 12, 18, 27,  3, 12, 10,
        27,  6,  3, 12,  7, 27,  5, 17, 12, 19, 21,  3, 24,  4])


In [ ]:
#Создадим нейроннную сеть RNN
#3 слоя:
#Embeding (30)
#RNN (hidden_dim=128)
# Полносвязный слой для предсказания буквы (28, то есть размер словаря)

class Network(torch.nn.Module):
    def __init__(self):
        super(Network, self).__init__()
        self.embedding = torch.nn.Embedding(len(AB_INDEX), 32)
        self.rnn = torch.nn.RNN(32, 128)
        self.out = torch.nn.Linear(128, len(AB_INDEX))

    def forward(self, sentences, state=None):
        x = self.embedding(sentences)
        x, s = self.rnn(x)
        return self.out(x)

In [ ]:
model = Network()
criterion = torch.nn.CrossEntropyLoss()  # лосс многоклассовой классификации
optimizer = torch.optim.SGD(model.parameters(), lr=.05)

In [ ]:
# Обучение модели
for ep in range(5):# количество эпох
    start = time.time()
    train_loss = 0.
    train_passed = 0

    for i in range(int(len(X) / 100)):
        # берём батч в 100 элементов
        batch = X[i * 100:(i + 1) * 100]
        X_batch = batch[:, :-1]
        Y_batch = batch[:, 1:].flatten()

        optimizer.zero_grad()
        answers = model.forward(X_batch)
        answers = answers.view(-1, len(AB_INDEX))
        loss = criterion(answers, Y_batch)
        train_loss += loss.item()

        loss.backward()
        optimizer.step()
        train_passed += 1

    print("Эпоха {}. Время обучения: {:.3f} сек, Ошибка обучения: {:.3f}".format(ep, time.time() - start, train_loss / train_passed))

Эпоха 0. Время обучения: 44.885 сек, Ошибка обучения: 1.737
Эпоха 1. Время обучения: 43.146 сек, Ошибка обучения: 1.656
Эпоха 2. Время обучения: 45.721 сек, Ошибка обучения: 1.642
Эпоха 3. Время обучения: 46.832 сек, Ошибка обучения: 1.634
Эпоха 4. Время обучения: 45.565 сек, Ошибка обучения: 1.626


In [ ]:
model.train(False)
for i in range(4):
    # возьмем произвольную строку из датасета
    idx = np.random.choice(len(X))
    # типа расшифруем ее
    pred = model.forward(X[idx].to(device))
    print(f'Пример № {i+1}')
    print('Исходный текст      :',''.join([alfabet[j] for j in y[idx] if j != 0]))
    print('Закодированный текст:',''.join([alfabet[j] for j in X[idx] if j != 0]))
    print('Расшифрованный текст:',''.join([alfabet[j] for j in torch.argmax(pred, dim=1) if j != 0]))
    print()

Пример № 1
Закодированный текст: szgkolbpkizekobzgykdspkvtyrkzqkypgkmebycktcwlyo
Исходный текст      : how dare you drown the king of new burns island
Расшифрованный текст: pesdkypkdkekdkpepkdkpkdkykkdekdkkkdpkpkkdykwdkk

Пример № 2
Закодированный текст: xlimpkcspkdszersdkdspkglbktykczedsplcdklctlkglckcz
Исходный текст      : maybe she thought the war in southeast asia was so
Расшифрованный текст: pyzpkdkpkdspekkpkdkpkdkdpdykdkekkpkykkdykyydsdkdke

Пример № 3
Закодированный текст: spbpkizekrzkblw s
Исходный текст      : here you go ralph
Расшифрованный текст: pkpkdkekdkedpdwkp

Пример № 4
Закодированный текст: tkdzzvklk pbczylwkoli
Исходный текст      : i took a personal day
Расшифрованный текст: ydseeplddwkpkekdwdkdk

